<a href="https://colab.research.google.com/github/MuhammadRafiq215/A-Level-Computer-Science-9618-Paper-42-Oct-Nov-2025/blob/main/CNN_Basics_Fashion_MNIST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CNN Basics — Image Classification with Convolutional Neural Networks

**Level:** Absolute beginner
**Dataset:** Fashion-MNIST (70,000 clothing images, 28x28 pixels, grayscale)
**Framework:** TensorFlow / Keras
**Runtime:** Works on CPU (no GPU needed) — about 3-5 minutes of training

---

### What you will learn

1. What a CNN is and *why* we need it for images
2. How a **convolution** actually works (we will do it by hand with NumPy)
3. What **pooling** does
4. How to load and prepare image data properly
5. How to build a CNN layer by layer in Keras
6. How to train it and read the learning curves
7. How to evaluate it — accuracy, confusion matrix, wrong predictions
8. How to **look inside** the network (feature maps and learned filters)

> **How to use this notebook:** run the cells one by one (Shift + Enter). Read the text before each cell. Do not skip the "Your turn" exercises at the end.

---
## 1. Setup

We import only what we need. Each library has one job:

| Library | Job |
|---|---|
| `numpy` | math on arrays (images are arrays of numbers) |
| `matplotlib` | drawing images and charts |
| `tensorflow` | building and training the neural network |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

In [ ]:
# Fix the random seed so your results match the notes (reproducibility)
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
print("TensorFlow version:", tf.__version__)
print("GPU available:", len(tf.config.list_physical_devices("GPU")) > 0)

In [ ]:
# Nicer default plots
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["image.cmap"] = "gray"
plt.rcParams["axes.grid"] = False

---
## 2. What is a CNN, and why not a normal neural network?

A grayscale image is just a **grid of numbers**. Each number is a pixel brightness from 0 (black) to 255 (white). Our images are 28x28, so each image is 784 numbers.

**The problem with a normal (dense) network:** it would flatten those 784 pixels into one long list and treat pixel #1 and pixel #500 as unrelated strangers. It throws away the most important fact about an image — that **nearby pixels belong together**. An edge, a sleeve, a shoe heel: these are *local* patterns.

**The CNN idea:** instead of looking at the whole image at once, slide a small window (e.g. 3x3) across the image and look for one specific little pattern. That sliding window is called a **filter** (or kernel).

### The three building blocks

**1. Convolution layer (`Conv2D`)**
Slides small filters over the image and produces a **feature map** — a new image showing *where* that pattern was found. Early filters learn simple things (edges, corners). Deeper filters combine those into complex things (textures, shapes, a whole sleeve).

**2. Pooling layer (`MaxPooling2D`)**
Shrinks the feature map by keeping only the strongest signal in each small block. This makes the network faster, and makes it less sensitive to *exactly* where the pattern was (a shoe is a shoe whether it sits 2 pixels left or right).

**3. Dense layer (`Dense`)**
After a few conv + pool rounds, we flatten everything into a list and use ordinary dense layers to make the final decision: "this is a sneaker".

### The typical pattern

```
INPUT image
  -> Conv2D  (find patterns)
  -> MaxPool (shrink)
  -> Conv2D  (find patterns made of patterns)
  -> MaxPool (shrink)
  -> Flatten (make it a list)
  -> Dense   (decide)
  -> OUTPUT: 10 probabilities, one per class
```

Notice the shape: the image gets **smaller** (28 -> 14 -> 7) while the number of filters gets **bigger** (32 -> 64). We trade spatial detail for pattern richness.

---
## 3. Loading the data

**Fashion-MNIST** is a drop-in replacement for the classic MNIST digits dataset, but with clothing items instead. It is perfect for learning because:

- it is small and downloads in seconds
- images are tiny (28x28) so training is fast on a CPU
- it is harder than digits, so your CNN's improvement over a dense network is actually visible

Keras ships a loader for it, so there is no manual downloading.

In [ ]:
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

In [ ]:
print("Training images:", x_train_full.shape)   # (60000, 28, 28)
print("Training labels:", y_train_full.shape)   # (60000,)
print("Test images:    ", x_test.shape)         # (10000, 28, 28)
print("Test labels:    ", y_test.shape)         # (10000,)

Read that shape carefully: `(60000, 28, 28)` means **60,000 images**, each **28 rows** by **28 columns**. There is no colour channel yet — we will add one in a moment.

In [ ]:
# The labels are numbers 0-9. These are the human names for them.
class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
]
print("Number of classes:", len(class_names))

In [ ]:
# What does one image actually look like as numbers?
print("Label:", y_train_full[0], "=", class_names[y_train_full[0]])
print("Pixel value range:", x_train_full.min(), "to", x_train_full.max())
print("Data type:", x_train_full.dtype)

In [ ]:
# Zoom into a 8x8 corner of one image so you can SEE that an image is just numbers
print(x_train_full[0][10:18, 10:18])

---
## 4. Look at the data before you model it

This is the step beginners skip, and it is the step professionals never skip. Always *look* at your data first.

In [ ]:
plt.figure(figsize=(4, 4))
plt.imshow(x_train_full[0])
plt.title(f"Label {y_train_full[0]}: {class_names[y_train_full[0]]}")
plt.colorbar()
plt.show()

In [ ]:
# A grid of 15 random images with their labels
plt.figure(figsize=(10, 5))
for i in range(15):
    plt.subplot(3, 5, i + 1)
    plt.imshow(x_train_full[i])
    plt.title(class_names[y_train_full[i]], fontsize=9)
    plt.axis("off")
plt.suptitle("Fashion-MNIST samples", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Is the dataset balanced? (equal number of each class?)
counts = np.bincount(y_train_full)
for name, c in zip(class_names, counts):
    print(f"{name:<12} {c}")

In [ ]:
plt.figure(figsize=(9, 4))
plt.bar(class_names, counts, color="steelblue")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Number of images")
plt.title("Class distribution in the training set")
plt.tight_layout()
plt.show()

Perfectly balanced — 6,000 images per class. That is good news: it means plain **accuracy** is a fair metric here. On an imbalanced dataset, accuracy would be misleading.

---
## 5. Preparing the data (preprocessing)

Two small but essential steps.

**Step A — Normalization.** Pixels are 0-255. Neural networks learn much faster and more stably when inputs are small numbers around 0-1, so we divide by 255. Skip this and your training will be slow and unstable.

**Step B — Add a channel dimension.** `Conv2D` expects images shaped `(height, width, channels)`. Colour images have 3 channels (red, green, blue). Ours are grayscale, so we have 1 channel — but that `1` still has to be there explicitly.

In [ ]:
# Step A: normalize to the range 0.0 - 1.0
x_train_full = x_train_full.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

print("New pixel range:", x_train_full.min(), "to", x_train_full.max())

In [ ]:
# Step B: (28, 28) -> (28, 28, 1)
x_train_full = np.expand_dims(x_train_full, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

print("Train shape now:", x_train_full.shape)
print("Test shape now: ", x_test.shape)

### Why we need a validation set

We have a training set and a test set. If we keep checking the test set while tuning the model, we start (accidentally) fitting to it, and our final score becomes a lie.

So we carve a **validation set** out of the training data:

- **Train set** — the model learns from this
- **Validation set** — we watch this during training to detect overfitting
- **Test set** — touched only ONCE, at the very end

In [ ]:
# Keep the last 6,000 training images aside for validation
x_val = x_train_full[54000:]
y_val = y_train_full[54000:]

x_train = x_train_full[:54000]
y_train = y_train_full[:54000]

print("Train:     ", x_train.shape)
print("Validation:", x_val.shape)
print("Test:      ", x_test.shape)

---
## 6. How a convolution really works (by hand, no Keras)

Before we let Keras hide the magic, let us do one convolution ourselves with pure NumPy. This is the single most important cell in the notebook.

**The recipe:**
1. Take a 3x3 filter (just 9 numbers).
2. Place it over the top-left 3x3 patch of the image.
3. Multiply each filter number with the pixel under it, and add all 9 results. That one number is the output pixel.
4. Slide the filter one pixel right, repeat. Then down a row, repeat. Until the image is covered.

That's it. A convolution is multiply-and-add, repeated across the image.

In [ ]:
# A famous hand-made filter: vertical edge detector (Sobel)
vertical_edge_filter = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1],
], dtype="float32")

print(vertical_edge_filter)

Why does this detect vertical edges? The left column is negative, the right column is positive. On a flat area, the negatives and positives cancel out and the result is ~0. But where brightness suddenly changes left-to-right, they do not cancel — and you get a large value. The filter "fires".

In [ ]:
def convolve2d(image, kernel):
    # Slide `kernel` over `image` and return the feature map (no padding)
    k = kernel.shape[0]
    out_h = image.shape[0] - k + 1
    out_w = image.shape[1] - k + 1
    output = np.zeros((out_h, out_w), dtype="float32")

    for row in range(out_h):
        for col in range(out_w):
            patch = image[row:row + k, col:col + k]   # the 3x3 window
            output[row, col] = np.sum(patch * kernel)  # multiply and add
    return output

In [ ]:
# Pick one image and run our own convolution on it
sample = x_train[0, :, :, 0]            # drop the channel dim for easy plotting
feature_map = convolve2d(sample, vertical_edge_filter)

print("Input size: ", sample.shape)
print("Output size:", feature_map.shape, "<- 2 pixels smaller (no padding)")

In [ ]:
plt.figure(figsize=(9, 4))

plt.subplot(1, 2, 1)
plt.imshow(sample)
plt.title("Original image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(np.abs(feature_map))
plt.title("After vertical-edge filter")
plt.axis("off")

plt.tight_layout()
plt.show()

Look at the right image: the vertical outlines of the garment glow, flat areas are dark. **That is a feature map.** The filter has turned "pixels" into "evidence of a vertical edge, and where it is".

Now the key insight: in a real CNN we **do not** choose these 9 numbers. The network **learns** them by itself during training, and it learns dozens of such filters at once.

Let us try a different filter so you can see the filter decides the feature.

In [ ]:
horizontal_edge_filter = np.array([
    [-1, -2, -1],
    [ 0,  0,  0],
    [ 1,  2,  1],
], dtype="float32")

h_map = convolve2d(sample, horizontal_edge_filter)

In [ ]:
plt.figure(figsize=(12, 4))
for i, (img, title) in enumerate([
    (sample, "Original"),
    (np.abs(feature_map), "Vertical edges"),
    (np.abs(h_map), "Horizontal edges"),
]):
    plt.subplot(1, 3, i + 1)
    plt.imshow(img)
    plt.title(title)
    plt.axis("off")
plt.tight_layout()
plt.show()

---
## 7. How max pooling works (by hand)

Max pooling with a 2x2 window: split the feature map into 2x2 blocks and keep only the **largest** value in each block. The result is half the width and half the height.

Why? Two reasons:
- **Fewer numbers** to process -> faster training, fewer parameters
- **Translation tolerance** -> we remember *that* a strong edge was in this neighbourhood, not its exact pixel

In [ ]:
def max_pool2d(feature_map, size=2):
    # Keep the maximum value in each `size` x `size` block
    h = feature_map.shape[0] // size
    w = feature_map.shape[1] // size
    output = np.zeros((h, w), dtype="float32")

    for row in range(h):
        for col in range(w):
            block = feature_map[row*size:(row+1)*size, col*size:(col+1)*size]
            output[row, col] = np.max(block)
    return output

In [ ]:
pooled = max_pool2d(np.abs(feature_map), size=2)

print("Before pooling:", feature_map.shape)
print("After pooling: ", pooled.shape, "<- exactly half")

In [ ]:
plt.figure(figsize=(9, 4))

plt.subplot(1, 2, 1)
plt.imshow(np.abs(feature_map))
plt.title(f"Feature map {feature_map.shape}")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(pooled)
plt.title(f"After 2x2 max pooling {pooled.shape}")
plt.axis("off")

plt.tight_layout()
plt.show()

Smaller, blockier — but the important structure survived. That is the whole point of pooling.

---
## 8. Building the CNN in Keras

Now we stack the real layers. Read the comments on every line — each one maps to a concept from above.

**The arguments explained:**

- `filters=32` — learn 32 different filters in this layer (so 32 feature maps come out)
- `kernel_size=(3, 3)` — each filter is a 3x3 window
- `activation="relu"` — ReLU turns negatives into 0. Without an activation, stacking layers would be pointless (a chain of straight lines is still a straight line)
- `padding="same"` — add a border of zeros so the output keeps the same height and width

We start with an explicit `Input` layer that declares the shape of one image: **height 28, width 28, channels 1**.

In [ ]:
model = keras.Sequential(name="basic_cnn")

In [ ]:
# Tell the model what one input image looks like
model.add(layers.Input(shape=(28, 28, 1)))

In [ ]:
# --- Block 1: find simple patterns (edges, corners) ---
model.add(layers.Conv2D(filters=32, kernel_size=(3, 3), activation="relu",
                        padding="same"))
model.add(layers.MaxPooling2D(pool_size=(2, 2)))     # 28x28 -> 14x14

In [ ]:
# --- Block 2: combine simple patterns into complex ones ---
model.add(layers.Conv2D(filters=64, kernel_size=(3, 3), activation="relu",
                        padding="same"))
model.add(layers.MaxPooling2D(pool_size=(2, 2)))     # 14x14 -> 7x7

In [ ]:
# --- Head: turn feature maps into a decision ---
model.add(layers.Flatten())                           # 7*7*64 = 3136 numbers
model.add(layers.Dense(64, activation="relu"))        # reasoning layer
model.add(layers.Dropout(0.3))                        # randomly switch off 30% of
                                                      # neurons while training,
                                                      # to fight overfitting
model.add(layers.Dense(10, activation="softmax"))     # 10 classes -> probabilities

In [ ]:
model.summary()

### Reading the summary

Follow the **Output Shape** column top to bottom and watch the image shrink while the channels grow:

```
(28, 28,  1)  input
(28, 28, 32)  conv  -> 32 feature maps
(14, 14, 32)  pool  -> half the size
(14, 14, 64)  conv  -> 64 feature maps
( 7,  7, 64)  pool  -> half again
(3136,)       flatten
(64,)         dense
(10,)         output
```

**Where do the parameters come from?** For the first conv layer:
`3 x 3 (filter) x 1 (input channel) x 32 (filters) + 32 (biases) = 320`

Notice that most parameters sit in the `Flatten -> Dense` connection. That is normal, and it is also where overfitting usually starts — which is exactly why we added Dropout there.

---
## 9. Compiling the model

Before training we must tell Keras three things:

- **optimizer** — *how* to update the weights. `adam` is a safe, fast default.
- **loss** — *what* to minimise. Our labels are plain integers (0-9), so we use `sparse_categorical_crossentropy`. (If labels were one-hot vectors, we would use `categorical_crossentropy` instead.)
- **metrics** — what *we* want to read. Accuracy is human-friendly; the loss is what the optimizer actually cares about.

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
print("Model compiled and ready to train.")

---
## 10. Training

Two words to know:

- **epoch** — one full pass over the whole training set
- **batch size** — how many images the model looks at before updating its weights (64 here)

Each epoch takes roughly 20-40 seconds on a CPU. Watch `val_accuracy` climb.

In [ ]:
EPOCHS = 8
BATCH_SIZE = 64

In [ ]:
history = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1,
)

---
## 11. Reading the learning curves

This is how you diagnose a model. Always plot these.

- Both curves still improving -> **train longer**
- Train accuracy high, validation accuracy stuck or falling -> **overfitting** (the model is memorising)
- Both low -> **underfitting** (model too small, or training too short)

In [ ]:
h = history.history
epochs_range = range(1, len(h["accuracy"]) + 1)

In [ ]:
plt.figure(figsize=(11, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, h["accuracy"], "o-", label="Train")
plt.plot(epochs_range, h["val_accuracy"], "s-", label="Validation")
plt.xlabel("Epoch"); plt.ylabel("Accuracy")
plt.title("Accuracy per epoch")
plt.legend(); plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(epochs_range, h["loss"], "o-", label="Train")
plt.plot(epochs_range, h["val_loss"], "s-", label="Validation")
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.title("Loss per epoch")
plt.legend(); plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
print(f"Final train accuracy:      {h['accuracy'][-1]*100:.2f}%")
print(f"Final validation accuracy: {h['val_accuracy'][-1]*100:.2f}%")
print(f"Gap (overfitting signal):  {(h['accuracy'][-1]-h['val_accuracy'][-1])*100:.2f} points")

---
## 12. The final exam: the test set

We touch the test set only now, once. This number is the honest answer to "how good is this model on images it has never seen?"

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)

print(f"Test loss:     {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy*100:.2f}%")

Around **90%** is a normal result for this simple CNN on Fashion-MNIST. For context: random guessing would give 10%, and a plain dense network usually stalls near 88%. The convolutions are earning their keep.

---
## 13. Looking at predictions

`model.predict` returns 10 probabilities per image. The predicted class is the one with the highest probability (`argmax`).

In [ ]:
probabilities = model.predict(x_test, verbose=0)
predictions = np.argmax(probabilities, axis=1)

print("Probabilities for the first test image:")
print(np.round(probabilities[0], 3))

In [ ]:
print("Predicted:", class_names[predictions[0]])
print("Actual:   ", class_names[y_test[0]])
print("Confidence:", f"{probabilities[0].max()*100:.1f}%")

In [ ]:
# 12 test images: green title = correct, red title = wrong
plt.figure(figsize=(12, 7))
for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(x_test[i, :, :, 0])
    correct = predictions[i] == y_test[i]
    plt.title(
        f"{class_names[predictions[i]]}\n(true: {class_names[y_test[i]]})",
        color="green" if correct else "red", fontsize=9,
    )
    plt.axis("off")
plt.suptitle("Predictions on unseen test images", fontsize=13)
plt.tight_layout()
plt.show()

### Where does the model fail?

Looking at your mistakes teaches you more than looking at your successes.

In [ ]:
wrong_indices = np.where(predictions != y_test)[0]
print("Wrong predictions:", len(wrong_indices), "out of", len(y_test))

In [ ]:
plt.figure(figsize=(12, 7))
for plot_i, idx in enumerate(wrong_indices[:12]):
    plt.subplot(3, 4, plot_i + 1)
    plt.imshow(x_test[idx, :, :, 0])
    plt.title(
        f"said: {class_names[predictions[idx]]}\ntrue: {class_names[y_test[idx]]}",
        color="red", fontsize=9,
    )
    plt.axis("off")
plt.suptitle("Mistakes — are they understandable?", fontsize=13)
plt.tight_layout()
plt.show()

---
## 14. Confusion matrix

A table where row = true class, column = predicted class. The diagonal is correct answers; everything off the diagonal is a confusion. It tells you *which* classes get mixed up — one overall accuracy number never does.

In [ ]:
confusion = tf.math.confusion_matrix(y_test, predictions).numpy()
print(confusion.shape)

In [ ]:
plt.figure(figsize=(8, 7))
plt.imshow(confusion, cmap="Blues")
plt.colorbar(label="Number of images")
plt.xticks(range(10), class_names, rotation=45, ha="right")
plt.yticks(range(10), class_names)
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.title("Confusion matrix")

for i in range(10):
    for j in range(10):
        plt.text(j, i, confusion[i, j], ha="center", va="center",
                 fontsize=7, color="white" if confusion[i, j] > 500 else "black")
plt.tight_layout()
plt.show()

In [ ]:
# Per-class accuracy, worst first
per_class = confusion.diagonal() / confusion.sum(axis=1)
for i in np.argsort(per_class):
    print(f"{class_names[i]:<12} {per_class[i]*100:5.1f}%")

Expect **Shirt** to be the worst class. That is not a bug — a shirt, a T-shirt, a pullover and a coat genuinely look alike at 28x28 pixels. Even humans struggle. Understanding *why* a model fails is a real skill.

---
## 15. Looking inside the trained CNN

Time to open the box. Two things worth seeing: the **filters** the network learned, and the **feature maps** it produces for a real image.

In [ ]:
first_conv = model.layers[0]
weights, biases = first_conv.get_weights()

print("Filter weights shape:", weights.shape)  # (3, 3, 1, 32) = 3x3 filter, 1 channel, 32 filters
print("Biases shape:        ", biases.shape)

In [ ]:
# The 32 learned 3x3 filters. Nobody designed these — training found them.
plt.figure(figsize=(9, 5))
for i in range(32):
    plt.subplot(4, 8, i + 1)
    plt.imshow(weights[:, :, 0, i])
    plt.axis("off")
plt.suptitle("The 32 filters learned by the first Conv2D layer", fontsize=12)
plt.tight_layout()
plt.show()

Light and dark patches arranged in stripes and blobs — the network discovered edge and texture detectors on its own, much like the Sobel filter we wrote by hand in section 6.

Now let us feed one image through and watch the feature maps.

In [ ]:
# A small model that stops after the first conv layer, so we can see its output.
# It reuses the SAME trained layer, so no retraining is needed.
activation_model = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    model.layers[0],
])

In [ ]:
test_image = x_test[0:1]                       # keep the batch dimension
activations = activation_model.predict(test_image, verbose=0)

print("Activation shape:", activations.shape)  # (1, 28, 28, 32)

In [ ]:
plt.figure(figsize=(4, 4))
plt.imshow(test_image[0, :, :, 0])
plt.title(f"Input: {class_names[y_test[0]]}")
plt.axis("off")
plt.show()

In [ ]:
plt.figure(figsize=(11, 6))
for i in range(32):
    plt.subplot(4, 8, i + 1)
    plt.imshow(activations[0, :, :, i], cmap="viridis")
    plt.axis("off")
plt.suptitle("32 feature maps from the first Conv2D layer", fontsize=12)
plt.tight_layout()
plt.show()

Each tile is one filter's answer to "where is my pattern in this image?" Some highlight the outline, some the inside texture, a few stay almost blank (that filter did not find its pattern here).

**This is how a CNN "sees":** not one picture, but 32 simultaneous pattern-evidence maps — and then 64 more built on top of those.

---
## 16. Saving the model

A trained model is worth saving — you can reload it later without retraining.

In [ ]:
model.save("fashion_cnn.keras")
print("Saved to fashion_cnn.keras")

In [ ]:
# Load it back and confirm it still works
reloaded = keras.models.load_model("fashion_cnn.keras")
loss, acc = reloaded.evaluate(x_test, y_test, verbose=0)
print(f"Reloaded model accuracy: {acc*100:.2f}%")

---
## 17. Summary — the ideas worth remembering

| Concept | One-line meaning |
|---|---|
| **Pixel** | a number; an image is a grid of numbers |
| **Filter / kernel** | a small window of weights that hunts for one pattern |
| **Convolution** | slide the filter, multiply and add -> a feature map |
| **Feature map** | a picture of *where* a pattern was found |
| **ReLU** | turn negatives into zero; this is what lets layers stack usefully |
| **Max pooling** | keep the strongest value per block; shrinks and adds tolerance |
| **Flatten** | turn feature maps into a flat list for the dense layers |
| **Dropout** | randomly switch neurons off while training, to fight overfitting |
| **Softmax** | turn raw scores into probabilities that sum to 1 |
| **Epoch** | one full pass over the training data |
| **Overfitting** | great on training data, poor on new data |

**The mental model:** a CNN is a stack of pattern detectors. Shallow layers see edges. Deeper layers see combinations of edges. The dense head turns that evidence into a label. And every one of those filters was *learned*, not programmed.

---
## 18. Your turn (exercises)

Do these in new cells. The goal is not to get a higher score — it is to build intuition for what each knob does.

**1. Break normalization.** Reload the data without dividing by 255 and retrain. What happens to the loss in epoch 1? Why?

**2. Remove the convolutions.** Build a model with only `Flatten -> Dense(64) -> Dense(10)`. Compare test accuracy and parameter count against the CNN. Which wins, and which is bigger?

**3. Filter count.** Change the first layer from 32 to 8 filters, then to 64. Watch accuracy, parameter count and training time.

**4. Kernel size.** Try `kernel_size=(5, 5)` instead of `(3, 3)`. More parameters per filter — does accuracy improve enough to justify it?

**5. Delete the pooling layers.** Keep both `Conv2D` layers but remove both `MaxPooling2D` layers. Check `model.summary()` — why did the parameter count explode?

**6. Force overfitting.** Remove `Dropout` and train for 25 epochs. Plot the curves. Can you see the exact epoch where validation loss turns upward?

**7. Add a third block.** Insert `Conv2D(128) + MaxPooling2D` before the Flatten. Deeper is not automatically better — did it help?

**8. Hand-made filters.** Write your own 3x3 kernel in `convolve2d` (try a blur: all nine values `1/9`). Predict the output before you plot it.

**9. Change the dataset.** Replace `keras.datasets.fashion_mnist` with `keras.datasets.mnist` (handwritten digits). The same code should run unchanged. Is accuracy higher or lower? Why?

**10. Your own image.** Draw a shoe in any paint app, save it, load it, convert to grayscale, resize to 28x28, invert if needed, normalize, and predict. Real-world images are messy — this exercise teaches more than the previous nine combined.

---

### Where to go next

- **Data augmentation** (`layers.RandomFlip`, `RandomRotation`) — create more training data for free
- **Batch normalization** — train deeper networks stably
- **Transfer learning** — reuse a network already trained on millions of images (this is what professionals actually do)
- **Colour images** — move to CIFAR-10 (32x32x3). Only the input shape changes; every idea in this notebook still applies.

*End of notebook. Happy learning!*